# HyperCrop — Plant Disease CNN Training

This notebook trains a MobileNetV2 model on the PlantVillage dataset to detect crop diseases from photos.

**Before running:**
1. Make sure you're using a GPU: `Runtime → Change runtime type → T4 GPU`
2. Run all cells top to bottom (`Runtime → Run all`)
3. At the end, download the `model.zip` file and extract it into `HyperCorp/backend/model/`

**Expected time:** ~20–30 minutes on T4 GPU

## Cell 1 — Check GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✓ GPU available: {gpus[0].name}')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('⚠ No GPU detected — go to Runtime → Change runtime type → T4 GPU')

print(f'TensorFlow version: {tf.__version__}')

## Cell 2 — Install Kaggle & set credentials

In [ ]:
!pip install kaggle -q

import os, json

# ── PASTE YOUR KAGGLE CREDENTIALS HERE ────────────────────────────────────
KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'   # e.g. 'ojaswalke356'
KAGGLE_KEY      = 'YOUR_KAGGLE_API_KEY'    # the KGAT_... token
# ──────────────────────────────────────────────────────────────────────────

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('✓ Kaggle credentials saved')

## Cell 3 — Download PlantVillage dataset

In [ ]:
import kaggle, zipfile, pathlib, shutil

DOWNLOAD_DIR = '/content/plantvillage_raw'
DATASET_DIR  = '/content/plantvillage'

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print('Downloading PlantVillage from Kaggle (~1.2 GB)...')
kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    'emmarex/plantdisease',
    path=DOWNLOAD_DIR,
    unzip=False
)

# Extract
zip_files = list(pathlib.Path(DOWNLOAD_DIR).glob('*.zip'))
print(f'Extracting {zip_files[0].name}...')
with zipfile.ZipFile(zip_files[0], 'r') as z:
    z.extractall(DOWNLOAD_DIR)

# Find the folder containing class subdirectories
dataset_root = None
for p in sorted(pathlib.Path(DOWNLOAD_DIR).rglob('*'), key=lambda x: len(x.parts), reverse=True):
    if p.is_dir():
        subdirs = [x for x in p.iterdir() if x.is_dir()]
        if len(subdirs) > 5:
            dataset_root = p
            break

if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
shutil.copytree(str(dataset_root), DATASET_DIR)

classes = list(pathlib.Path(DATASET_DIR).iterdir())
print(f'✓ Dataset ready — {len(classes)} classes found')
print('Sample classes:', [c.name for c in classes[:5]])

## Cell 4 — Build data pipelines

In [ ]:
import numpy as np

IMG_SIZE         = (224, 224)
BATCH_SIZE       = 32
VALIDATION_SPLIT = 0.15
TEST_SPLIT       = 0.10
SEED             = 42

full_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    image_size=IMG_SIZE,
    batch_size=None,
    label_mode='categorical',
    shuffle=True,
    seed=SEED,
)

class_names = full_ds.class_names
n_classes   = len(class_names)
total       = sum(1 for _ in full_ds)

n_val   = int(total * VALIDATION_SPLIT)
n_test  = int(total * TEST_SPLIT)
n_train = total - n_val - n_test

print(f'Total images : {total}')
print(f'Classes      : {n_classes}')
print(f'Train / Val / Test : {n_train} / {n_val} / {n_test}')

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1),
], name='augmentation')

AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, augment_fn=None):
    if augment_fn:
        ds = ds.map(lambda x, y: (augment_fn(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = prepare(full_ds.take(n_train), augment)
val_ds   = prepare(full_ds.skip(n_train).take(n_val))
test_ds  = prepare(full_ds.skip(n_train + n_val))

print('✓ Data pipelines ready')

## Cell 5 — Build MobileNetV2 model

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base.trainable = False

inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3), name='image_input')
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(n_classes, activation='softmax', name='predictions')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print(f'✓ Model built — {model.count_params():,} parameters')
print(f'  Trainable : {sum(tf.size(v).numpy() for v in model.trainable_variables):,}')

## Cell 6 — Phase 1: Train head (base frozen)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss', verbose=1),
]

print('Phase 1 — training head with frozen base (5 epochs max)...')
h1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks,
)
print(f'✓ Phase 1 done — val_accuracy: {max(h1.history["val_accuracy"]):.4f}')

## Cell 7 — Phase 2: Fine-tune top layers

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print('Phase 2 — fine-tuning top 30 layers (10 epochs max)...')
h2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
)
print(f'✓ Phase 2 done — val_accuracy: {max(h2.history["val_accuracy"]):.4f}')

## Cell 8 — Evaluate on test set

In [ ]:
loss, acc = model.evaluate(test_ds, verbose=1)
print(f'\n✓ Test accuracy : {acc * 100:.2f}%')
print(f'  Test loss     : {loss:.4f}')

## Cell 9 — Save model + class index

In [ ]:
import json

os.makedirs('/content/model', exist_ok=True)

model.save('/content/model/plant_disease_model.h5')
print('✓ Model saved to /content/model/plant_disease_model.h5')

class_idx = {str(i): name for i, name in enumerate(class_names)}
with open('/content/model/class_indices.json', 'w') as f:
    json.dump(class_idx, f, indent=2)
print(f'✓ Class index saved ({len(class_names)} classes)')

# Save training history
history = {}
for k in h1.history:
    history[k] = h1.history[k] + h2.history[k]
history['test_accuracy'] = float(acc)
history['test_loss']     = float(loss)
with open('/content/model/training_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] if isinstance(vals, list) else vals
               for k, vals in history.items()}, f, indent=2)
print('✓ Training history saved')

print(f'\nFinal test accuracy: {acc * 100:.2f}%')

## Cell 10 — Convert to ONNX

Converts the trained model to ONNX format so it can run locally on **any Python version** (including 3.14) using `onnxruntime` — no TensorFlow needed on your machine.

In [ ]:
!pip install tf2onnx onnxruntime -q
import tf2onnx, onnx

print('Converting model to ONNX...')
input_signature = [tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image_input')]
onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=input_signature, opset=13)
onnx.save(onnx_model, '/content/model/plant_disease_model.onnx')
print('✓ ONNX model saved to /content/model/plant_disease_model.onnx')

# Quick sanity check
import onnxruntime as ort
import numpy as np
sess = ort.InferenceSession('/content/model/plant_disease_model.onnx')
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
out = sess.run(None, {'image_input': dummy})
print(f'✓ ONNX sanity check passed — output shape: {out[0].shape}')

## Cell 11 — Download model as zip

Downloads everything — ONNX model + class index. Extract into `HyperCorp/backend/model/`.

```
HyperCorp/backend/model/
    plant_disease_model.onnx   ← main file used by the app
    class_indices.json
    training_history.json
```

You do NOT need the `.h5` file on your local machine.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/model', 'zip', '/content/model')
files.download('/content/model.zip')
print('✓ Download started')